### Pacotes importados

In [1]:
using LinearAlgebra
using Printf

is_positive_definite (generic function with 1 method)

## Chapter 9: Quadratic problems

![image.png](attachment:0deae26d-d155-4346-99c5-a223cd48dbe4.png)

### Algorithm 9.1: quadratic problems: direct solution

![image.png](attachment:23e99710-51b3-48cc-8751-58262b67d9c5.png)

Example 9.8: $Q=\left(\begin{array}{cccc} 1& 1 & 1 & 1 \\ 1 & 2 & 2 & 2 \\ 1 & 2 & 3 & 3 \\ 1 & 2 & 3 & 4\end{array}\right)$, $b=\left(\begin{array}{c}-4 \\ -7 \\ -9 \\ -10\end{array}\right)$. First, confirm Q is P.D.

In [5]:
# Helper function for the quadratic objective
# f(x) = 1/2 x'Qx + b'x
quadratic_value(Q, b, x) = 0.5 * dot(x, Q * x) + dot(b, x)

# Positive definiteness check
function is_positive_definite(Q)
    try
        cholesky(Symmetric(Q))
        return true
    catch
        return false
    end
end

# Data from Example 9.8
Q = [
    1.0 1.0 1.0 1.0;
    1.0 2.0 2.0 2.0;
    1.0 2.0 3.0 3.0;
    1.0 2.0 3.0 4.0
]

b = [-4.0, -7.0, -9.0, -10.0]

println("Q is positive definite? ", is_positive_definite(Q))
println("Eigenvalues of Q: ", round.(eigvals(Symmetric(Q)), digits=6))

# Direct solution for min 1/2 x'Qx + b'x
# FONC: Qx + b = 0 -> x* = -Q^{-1}b
x_star = -Q \ b
f_star = quadratic_value(Q, b, x_star)

println("\nDirect solution:")
println("x* = ", round.(x_star, digits=6))
println("f(x*) = ", round(f_star, digits=6))
println("gradient Qx + b = ", round.(Q * x_star + b, digits=6))


Q is positive definite? true
Eigenvalues of Q: [0.283119, 0.426022, 1.0, 8.290859]

Direct solution:
x* = [1.0, 1.0, 1.0, 1.0]
f(x*) = -15.0
gradient Qx + b = [0.0, 0.0, 0.0, 0.0]


### Algorithm 9.2: Conjugate gradient method

![image.png](attachment:6c5c50dd-e28c-492b-8044-8a85d1e73f7d.png)

Run the algorithm from $x_0=\left(\begin{array}{c}5 \\ 5 \\ 5 \\ 5 \end{array}\right)$.

In [6]:
function conjugate_gradient(Q, b, x0; tol=1e-8, max_iter=100)
    if !is_positive_definite(Q)
        error("Matrix Q must be positive definite for the conjugate gradient method.")
    end

    x = copy(x0)
    g = Q * x + b              # gradient
    d = -g                     # first descent direction

    println("Conjugate Gradient Method")
    println("Initial x = ", x)
    println()

    for k in 0:max_iter
        fval = quadratic_value(Q, b, x)
        gnorm = norm(g)

        @printf("Iter %2d | f(x) = %.8f | ||grad|| = %.8e | x = %s\n",
                k, fval, gnorm, string(round.(x, digits=6)))

        if gnorm < tol
            break
        end

        ak = -dot(g, d) / dot(d, Q * d)
        x_new = x + ak * d
        g_new = Q * x_new + b
        bk = dot(g_new, g_new) / dot(g, g)
        d = -g_new + bk * d

        x = x_new
        g = g_new
    end

    return x
end

x0 = [5.0, 5.0, 5.0, 5.0]
x_cg = conjugate_gradient(Q, b, x0)

println("\nFinal result:")
println("x* = ", round.(x_cg, digits=6))
println("f(x*) = ", round(quadratic_value(Q, b, x_cg), digits=6))


Conjugate Gradient Method
Initial x = [5.0, 5.0, 5.0, 5.0]

Iter  0 | f(x) = 225.00000000 | ||grad|| = 6.27375486e+01 | x = [5.0, 5.0, 5.0, 5.0]
Iter  1 | f(x) = -12.66715758 | ||grad|| = 2.08593555e+00 | x = [3.067747, 1.618557, 0.65243, 0.169367]
Iter  2 | f(x) = -14.90697422 | ||grad|| = 2.77585326e-01 | x = [1.496896, 0.610224, 0.847993, 1.215542]
Iter  3 | f(x) = -14.99834915 | ||grad|| = 3.12027374e-02 | x = [1.028064, 0.938093, 1.074288, 0.965332]
Iter  4 | f(x) = -15.00000000 | ||grad|| = 7.16436260e-12 | x = [1.0, 1.0, 1.0, 1.0]

Final result:
x* = [1.0, 1.0, 1.0, 1.0]
f(x*) = -15.0


An error is triggered if the matrix is not definite positive. Here, $Q=\left(\begin{array}{cccc} 1& 2 & 3 & 4 \\ 5 & 6 & 7 & 8 \\ 9 & 10 & 11 & 12 \\ 13 & 14 & 15 & 16\end{array}\right)$

In [4]:
# Example with a matrix that is not positive definite
Q_bad = [
    1.0  2.0  3.0  4.0;
    5.0  6.0  7.0  8.0;
    9.0 10.0 11.0 12.0;
    13.0 14.0 15.0 16.0
]

println("Q_bad is positive definite? ", is_positive_definite(Q_bad))

try
    x_bad = conjugate_gradient(Q_bad, b, x0)
catch err
    println("Expected error: ", err)
end


Q_bad is positive definite? false
Expected error: ErrorException("Matrix Q must be positive definite for the conjugate gradient method.")
